# Fine-tune Qwen2.5-Coder-1.5B — Русский учитель кода
Датасет встроен (base64). Нажми **Runtime → Run all**
Нужен GPU: **Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q torch transformers accelerate peft bitsandbytes trl unsloth
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python 2>/dev/null || true
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import json, base64, random
encoded = '...'
data = json.loads(base64.b64decode(encoded).decode('utf-8'))
random.shuffle(data)
print(f'Загружено {len(data)} примеров')

In [ ]:
def fmt(ex):
    s = 'Ты — опытный учитель программирования. Отвечаешь на русском, с примерами кода.'
    u = ex['instruction']
    if ex.get('input'): u += '\n' + ex['input']
    return {'text': f"<|im_start|>system\n{s}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{ex['output']}<|im_end|>"}
formatted = [fmt(ex) for ex in data]
print(formatted[0]['text'][:120])

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-Coder-1.5B-Instruct',
    max_seq_length=2048, dtype=torch.bfloat16, load_in_4bit=True,
)
print('Модель загружена')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_rslora=True,
)
print(f'Trainable: {model.num_parameters(only_trainable=True):,}')

In [ ]:
import datasets
def tok(ex): return tokenizer(ex['text'], truncation=True, max_length=2048)
ds = datasets.Dataset.from_list(formatted).map(tok, batched=False)
print(f'Примеров: {len(ds)}')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=TrainingArguments(
        output_dir='/content/qwen-coder-finetuned',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=2e-4,
        bf16=True, logging_steps=10, save_steps=100,
        save_total_limit=2, remove_unused_columns=False,
        report_to='none',
    ),
)
trainer.train()
print('Обучение завершено')

In [ ]:
model.save_pretrained('/content/qwen-coder-lora')
tokenizer.save_pretrained('/content/qwen-coder-lora')
print('LoRA сохранена')

In [ ]:
from peft import PeftModel
base, _ = FastLanguageModel.from_pretrained(
    'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    load_in_4bit=False, dtype=torch.bfloat16,
)
lora = PeftModel.from_pretrained(base, '/content/qwen-coder-lora')
merged = lora.merge_and_unload()
merged.save_pretrained('/content/qwen-coder-merged')
tokenizer.save_pretrained('/content/qwen-coder-merged')
print('Модель объединена')

In [ ]:
import os
if not os.path.exists('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install -q /content/llama.cpp 2>/dev/null || true
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/qwen-coder-merged \
    --outfile /content/qwen-coder-russian.gguf \
    --outtype q4_k_m
print(f'GGUF: {os.path.getsize("/content/qwen-coder-russian.gguf") / 1024**2:.0f} MB')
import shutil
shutil.make_archive('/content/qwen-coder', 'zip', '/content', 'qwen-coder-russian.gguf')

In [ ]:
from google.colab import files
files.download('/content/qwen-coder.zip')
print('Готово! Распакуй .gguf, запусти llama-server')